# CrossCodeEval adapter demo

Runs **one real CrossCodeEval example** through the full, unmodified pipeline:
loads the example (task_id/repository/file/prompt/groundtruth), clones and indexes
the referenced GitHub repo on demand, nominates candidates (BM25 + symbol +
dependency), lets Qwen pick useful ones, then StarCoder generates the completion.

Only `prompt` and `file` are ever passed to retrieval/selection/generation --
`groundtruth`/`right_context` are extracted but never fed downstream.

Uses the Hugging Face backends (not Ollama) -- proven more reliable on Colab.

**Before running**: Runtime -> Change runtime type -> select a GPU (T4 is fine).

In [ ]:
# Confirm a GPU is actually visible to this session -- if this errors or
# shows no GPU, Runtime -> Manage sessions -> terminate all, then reconnect.
!nvidia-smi

## 1. Get the project code

In [ ]:
%cd /content
import shutil, os
if os.path.exists("repo-code-completion"):
    shutil.rmtree("repo-code-completion")
!git clone "https://github.com/Robertkiza0/repo-code.git" repo-code-completion
%cd /content/repo-code-completion
!git log --oneline -3

## 2. Install dependencies

In [ ]:
!pip install -q tree-sitter tree-sitter-python tree-sitter-java tree-sitter-typescript tree-sitter-c-sharp rank-bm25 requests python-Levenshtein
!pip install -q torch transformers accelerate bitsandbytes

## 3. Hugging Face login

Add your token as a Colab Secret named `HF_TOKEN` (padlock icon, left sidebar)
before running this -- never pasted directly into the notebook.

In [ ]:
from huggingface_hub import login

try:
    from google.colab import userdata
    login(token=userdata.get("HF_TOKEN"))
except Exception:
    login()  # prompts for the token interactively (input is hidden)

## 4. Load the example and its repository index

First run clones + indexes the referenced repo (cached under `data/cceval/` for
later runs). Looks up the example by `TASK_ID` rather than a positional index;
set it to any task_id from the 20-example sample (or pass a different
`jsonl_path` to draw from the full dataset -- requires the full CrossCodeEval
archive to be extracted, see the main README).

In [ ]:
from evaluation.cceval_adapter import DEFAULT_JSONL, find_example_index_by_task_id, load_cceval_example, locate_repo_index

TASK_ID = "project_cc_python/62"

EXAMPLE_INDEX = find_example_index_by_task_id(DEFAULT_JSONL, TASK_ID)
example = load_cceval_example(index=EXAMPLE_INDEX)
chunks = locate_repo_index(example["repository"])  # clones + indexes on first run, cached after
print("task_id:   ", example["task_id"])
print("repository:", example["repository"])
print("file:      ", example["file"])
print(f"{len(chunks)} chunks indexed")

## 5. Load the models (run this ONCE per session)

Loads Qwen (selection) and StarCoder (generation) onto the GPU. **Do not
re-run this cell** unless you've just restarted the kernel -- re-running it
without restarting loads a second copy of both models on top of whatever's
already resident, which alone can exhaust a T4's 15GB VRAM. If you need to
reload (e.g. to change `load_in_4bit` or the model), restart the kernel
first.

In [ ]:
from selection.backends import HuggingFaceBackend
from selection.llm_selector import LLMSelector
from generation.backends import HuggingFaceGenerationBackend
from generation.generator import CompletionGenerator

selector = LLMSelector(chunks, backend=HuggingFaceBackend())  # Qwen2.5-Coder-7B-Instruct, 4-bit
generator = CompletionGenerator(chunks, backend=HuggingFaceGenerationBackend())  # StarCoder2-3b, 4-bit
print("models loaded")

## 6. Verify isolation, run, and print (safe to re-run repeatedly)

Reuses the already-loaded `selector`/`generator` from section 5 -- this cell
does no model loading itself, so re-running it doesn't touch GPU memory
allocation the way section 5 does. Explicitly asserts/logs that groundtruth
was never used for retrieval/selection/generation (only for the final
comparison), and sanity-checks that the completion is genuinely generated
text rather than a leaked chunk source.

In [ ]:
import time
from evaluation.cceval_adapter import verify_and_run_task, print_experiment_log

t0 = time.time()
result = verify_and_run_task(TASK_ID, selector=selector, generator=generator)
print(f"took {time.time() - t0:.1f}s\n")

print_experiment_log(chunks, result)

## 7. Scale to 20 tasks (do not run a larger experiment yet)

Runs the same unmodified pipeline across all 20 tasks in the sample file
(reuses `selector`/`generator` from section 5 -- no reload). Only 4 unique
repos are referenced across all 20 tasks (already-cached `turboderp/exllama`
covers 14 of them), so this clones at most 3 new repos. A failing task is
recorded with its error and does not stop the run. Saves
`results/cceval_20_results.jsonl` and `results/cceval_20_summary.json`.

In [ ]:
from evaluation.experiment import preflight_check, print_summary_table, print_task_table, run_experiment

# Verify the 20 tasks + their repository/index mapping before running anything.
mapping = preflight_check(DEFAULT_JSONL, 20)
for m in mapping:
    print(f"  [{m['index']:2d}] {m['task_id']:<26} repo={m['repository']:<30} file={m['file']}")

In [ ]:
outcome = run_experiment(n_tasks=20, selector=selector, generator=generator)

print_summary_table(outcome["summary"])
print()
print_task_table(outcome["results"])

## 8. Inspect selection validity (before running another experiment)

Part A works on the just-saved `outcome["results"]` alone: verifies every
selected candidate label is actually a member of that task's own candidate
pool (making LLMSelector's built-in hallucination-filtering explicit and
checkable). Flags `selected_count == 0` cases -- a saved result alone can't
tell an intentional empty selection apart from a parsing failure.

Part B re-runs retrieval + selection ONLY (no generation, so it's fast) for
specific zero-selection task_ids, capturing Qwen's raw text response to
resolve that ambiguity definitively. Edit `ZERO_SELECTION_TASK_IDS` to match
whatever section 7 actually reported as `selected == 0` in your run.

In [ ]:
# Part A: structural validity check on the already-saved results.
from evaluation.experiment import check_selection_validity, print_selection_validity_table

validity_rows = check_selection_validity(outcome["results"])
print_selection_validity_table(validity_rows)

In [ ]:
# Part B: re-run selection only (fast, no generation) for zero-selection
# tasks to see Qwen's actual raw response and resolve the ambiguity.
from evaluation.experiment import inspect_task_selection, print_task_selection_diagnosis

ZERO_SELECTION_TASK_IDS = ["project_cc_python/67", "project_cc_python/75"]

for task_id in ZERO_SELECTION_TASK_IDS:
    diagnosis = inspect_task_selection(task_id, selector=selector)
    print_task_selection_diagnosis(diagnosis)
    print("=" * 70)

## 9. Re-verify the 3 zero-selection tasks with the fixed parser

The selection-response parser didn't strip markdown code fences before
parsing, so `project_cc_python/67`'s `` ```json\n[]\n``` `` and
`/75`'s fenced `{"selected_chunk_ids": []}` were both misclassified as
parse failures -- they're valid JSON. Fixed (parser only; retrieval,
selection, generation, and evaluation logic untouched). Re-running all 3
tasks that had `selected == 0` in section 7 (`67`, `75`, `56`) to confirm.
Selection-only (no generation) -- fast. **Not re-running the 20-task
experiment yet.**

In [ ]:
# Reload evaluation.experiment / selection.llm_selector -- both already
# imported earlier this session, so a plain `!git pull` alone won't pick up
# the fix without this. Reloading the module alone isn't enough for the
# already-constructed `selector` object though (it stays bound to the old
# class code) -- rebuild it, reusing its existing backend so the GPU model
# itself doesn't reload.
import importlib
import selection.llm_selector, evaluation.cceval_adapter, evaluation.experiment
importlib.reload(selection.llm_selector)
importlib.reload(evaluation.cceval_adapter)
importlib.reload(evaluation.experiment)

selector = selection.llm_selector.LLMSelector(chunks, backend=selector.backend)

from evaluation.experiment import inspect_task_selection, print_parse_diagnosis

DIAGNOSTIC_TASK_IDS = ["project_cc_python/67", "project_cc_python/75", "project_cc_python/56"]

for task_id in DIAGNOSTIC_TASK_IDS:
    diagnosis = inspect_task_selection(task_id, selector=selector)
    print_parse_diagnosis(diagnosis)
    print("=" * 70)

## 10. Re-run the 20-task experiment with the fixed parser (v2)

Same unmodified retrieval / candidate nomination / candidate ordering / Qwen
selection prompt / StarCoder generation / evaluation metrics as section 7 --
only the selection-response parser changed. Empty selection (`[]`) is kept
as a valid, unforced outcome. Saves separately as
`results/cceval_20_results_v2.jsonl` and `results/cceval_20_summary_v2.json`
-- section 7's original files are untouched.

In [ ]:
# Make sure the fixed parser is actually loaded in this session (harmless
# if section 9 already did this).
import importlib
import selection.llm_selector, evaluation.cceval_adapter, evaluation.experiment
importlib.reload(selection.llm_selector)
importlib.reload(evaluation.cceval_adapter)
importlib.reload(evaluation.experiment)
selector = selection.llm_selector.LLMSelector(chunks, backend=selector.backend)

from evaluation.experiment import run_experiment_v2, print_summary_table, print_task_table

outcome_v2 = run_experiment_v2(n_tasks=20, selector=selector, generator=generator)

print_summary_table(outcome_v2["summary"])
print()
print_task_table(outcome_v2["results"])

## 11. No-selection baseline (same 20 tasks, skip Qwen entirely)

Same retrieval (BM25 + symbol + dependency, deduplicated, capped at 12) and
same StarCoder generation as sections 7/10 -- the **only** difference: every
nominated candidate is handed straight to the generator, `LLMSelector`/Qwen
is never constructed or called. Reuses the already-loaded `generator` from
section 5 -- **no need to have run section 5's `selector` line, or any Qwen
cell, before this**. Groundtruth is never passed to retrieval or generation,
only used afterward for evaluation. Saves separately as
`results/cceval_20_baseline.jsonl` and
`results/cceval_20_baseline_summary.json` -- does not touch or overwrite
sections 7/10's own result files.

In [ ]:
from evaluation.baseline import (
    print_baseline_isolation_flags,
    print_baseline_summary_table,
    print_baseline_task_table,
    run_baseline_experiment,
)

print_baseline_isolation_flags()
print()

outcome_baseline = run_baseline_experiment(generator, n_tasks=20)

print_baseline_summary_table(outcome_baseline["summary"])
print()
print_baseline_task_table(outcome_baseline["results"])

## 12. min2_selection ablation (same 20 tasks, floor Qwen's selection at 2)

Same retrieval / candidate nomination / candidate ordering / Qwen selection
prompt / StarCoder generation / evaluation as sections 7/10 -- the **only**
difference: after Qwen's (validated) selection, if it picked fewer than 2
candidates, the shortfall is filled using the next-highest-ranked
candidates from the **same already-computed candidate pool** (no new
retrieval or ranking). Qwen's own picks are always kept, never discarded.

Tests whether the no-selection-vs-Qwen-selection gap you just found is
mainly **over-pruning** (recoverable by a floor) or Qwen picking the wrong
chunks outright. Reuses `selector`/`generator` from section 5. Saves
separately as `results/cceval_20_min2_selection.jsonl` and
`results/cceval_20_min2_selection_summary.json` -- does not touch or
overwrite sections 7/10/11's own result files.

In [ ]:
import json
from pathlib import Path
from evaluation.min2_selection import (
    print_min2_isolation_flags,
    print_min2_summary_table,
    print_min2_task_table,
    print_three_way_comparison,
    run_min2_selection_experiment,
)

print_min2_isolation_flags()
print()

outcome_min2 = run_min2_selection_experiment(selector, generator, n_tasks=20)

print_min2_summary_table(outcome_min2["summary"])
print()
print_min2_task_table(outcome_min2["results"])

# Three-way comparison against the baseline and plain Qwen-selection
# summaries already saved to disk (works even if this is a fresh session
# that only ran sections 1/2/5/11/12).
baseline_summary = json.loads(Path("results/cceval_20_baseline_summary.json").read_text())
qwen_summary = json.loads(Path("results/cceval_20_summary_v2.json").read_text())

print()
print_three_way_comparison(baseline_summary, qwen_summary, outcome_min2["summary"])